# 38. 箱线图（boxplot）

<!-- module-learning-arc:start -->
> **Matplotlib 模块主线｜第 7 / 12 步：观察关系、分布与构成**
>
> **持续应用背景：** 制作经营周会一页报告：把趋势、比较、分布和异常证据组织成有主次、可直接用于会议的静态页面。
>
> **承接上一阶段：** 直方图（hist）  →  **本章任务：** 箱线图（boxplot）  →  **下一步：** 面积图（fill_between / stackplot）
>
> **大作业连接：** 本章练习将成为《经营周会一页报告》的一部分，最终需要从周会问题出发选择互补图形，完成视觉层级、注释审阅与独立导出。
<!-- module-learning-arc:end -->


## 本章场景

拿到一列或多列数字（比如订单金额、配送时长、各品类销量）时，我们最关心的往往不只是"平均是多少"，而是"数据大致集中在哪、波动大不大、有没有特别离谱的极端值"。



## 本章目标

学完本章，你将能够：

- **理解**：理解「箱线图（boxplot）」的适用场景、数据结构要求，以及它想帮你读出的规律。
- **操作**：能按参数用相应绘图接口画出「箱线图（boxplot）」，并做必要的美化、注释与导出。
- **迁移**：能换一份真实经营数据，独立画出同类型的「箱线图（boxplot）」并读出其中的结论。


## 38.1 适用场景

**背景引入**：拿到一列或多列数字（比如订单金额、配送时长、各品类销量）时，我们最关心的往往不只是"平均是多少"，而是"数据大致集中在哪、波动大不大、有没有特别离谱的极端值"。箱线图正是为回答这一类问题而生的经典图表：一行 `ax.boxplot()` 就能把样本的中心、四分位离散范围和异常观察同时展示出来，非常适合在动手建模或写结论之前，先用一幅图快速看清数据的分布轮廓。 打个比方：箱线图就像给这批数据做一次「体检」——中间那口箱子是大多数样本所在的正常范围，上下两条须延伸到正常值的两端，箱子外单独的点就是要多看两眼的高异常值；一行代码就把「集中在哪、波动多大、有没有离谱值」一次讲清楚。


## 38.2 图表与参数速查

先用这张表建立本章的方法地图；每一行后面都有对应的独立示例或练习。

| 类别 | 常用方法或写法 | 主要用途 | 需要特别注意 |
| --- | --- | --- | --- |
| 基础图表 | `plt.subplots()`、`ax.boxplot()`、`ax.set()`、`ax.grid()` | 比较一个或多个数值样本的中心、离散程度与异常观察。 | 把箱体高度理解为样本量 |
| 进阶变体 | `np.random.default_rng()`、`rng_box.normal()`、`plt.subplots()`、`ax.boxplot()` | 在基础图表上增加分组、注释、布局或交互 | 机械删除所有须外点 |
| 关键参数 | `whis` | 须范围 | 把箱体高度理解为样本量 |
| 关键参数 | `showmeans` | 均值 | 机械删除所有须外点 |
| 关键参数 | `notch` | 中位数缺口 | 小样本仍只看箱线摘要 |
| 关键参数 | `patch_artist` | 填充箱体 | 把箱体高度理解为样本量 |


## 38.3 准备可复现数据

先完成导入和数据准备，后续单元格只负责一种图表或一种分析动作。


<!-- math-foundation:chapter-38 -->
### 数学推导｜箱线图的四分位数与异常界限

> 阅读方法：先跟着步骤理解每个量怎样产生，再看最后的可计算形式；不需要脱离业务场景死记公式。

**第 1 步｜用分位数切分排序数据。** $Q_1$、$Q_2$、$Q_3$ 分别对应累计比例 25%、50%、75%。

**第 2 步｜中间一半数据的跨度是**

$$
IQR=Q_3-Q_1
$$

**第 3 步｜把箱体向两侧延伸 1.5 个 IQR。** 下、上界分别为 $L=Q_1-1.5IQR$、$U=Q_3+1.5IQR$。箱线图的“须”通常落到界内最远的实际观测，而不是直接画到 $L$、$U$。

**把上面的关系收束为本章计算式：**

$$
IQR=Q_3-Q_1,\qquad [L,U]=[Q_1-1.5IQR,\ Q_3+1.5IQR]
$$

**符号解释：** $Q_1$、$Q_3$ 是第一和第三四分位数，IQR 描述中间 50% 数据的跨度。

**代码对应：** 用 `quantile([.25, .5, .75])` 复核图中的箱体和中位数。

**使用边界：** 落在界限外的是统计异常点，不等于错误数据，更不能自动删除。


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

# 中文字体支持：由平台运行时自动配置
# 说明：Matplotlib 默认字体不含中文字形，中文会显示成方框。
#      本平台在运行每个绘图 cell 前，会自动注册可用的中文字体并设置
#      font.sans-serif / axes.unicode_minus，因此这里不需要手动 import
#      或 addfont，直接使用即可。

# 1️⃣ 数据导入：读取订单数据（指定列类型降低内存、加快分组）
transactions = pd.read_csv(
    "/datasets/uci_online_retail_200k.csv",
    parse_dates=["InvoiceDate"],
    dtype={"Country": "category"},
)
print(f"数据规模：{len(transactions):,} 行 × {transactions.shape[1]} 列")


In [ ]:
# 2️⃣ 特征工程：构造分析所需字段与聚合结果
transactions["amount"] = transactions["Quantity"] * transactions["UnitPrice"]

# 有效订单：数量与单价均为正（退货/取消行不参与月度统计）
completed = transactions.query("Quantity > 0 and UnitPrice > 0")
completed["month"] = completed["InvoiceDate"].dt.to_period("M").astype("string")

# 月度聚合：销售额（元）与订单数
monthly_summary = completed.groupby("month").agg(
    sales=("amount", "sum"), orders=("InvoiceNo", "nunique")
)
months = monthly_summary.index.to_numpy()
sales = (monthly_summary["sales"] / 10_000).to_numpy()
orders = monthly_summary["orders"].to_numpy()
profit = sales * 0.18  # 简化假设：利润约为销售额的 18%

# 区域构成：销售额前 4 国，统计「销售 vs 退货」两部分（单位：万元）
top = completed.groupby("Country")["amount"].sum().nlargest(4).index
rows = transactions[transactions["Country"].isin(top)].copy()
rows["flow"] = np.where(rows["Quantity"] > 0, "销售", "退货")
regional = (
    pd.crosstab(rows["Country"], rows["flow"], values=rows["amount"].abs(), aggfunc="sum")
    / 10_000
).fillna(0)
regions = regional.index.to_numpy()
online = regional["销售"].to_numpy()
offline = regional["退货"].to_numpy()

# 固定随机种子抽样 2000 条，供分布图使用，保证每次运行结果一致
samples = completed["amount"].sample(2_000, random_state=25).to_numpy()
print(f"有效订单：{len(completed):,} 行")


## 38.4 基础图表

先保留必要的编码：位置、颜色或大小。图表标题、坐标轴和单位应能让读者脱离代码理解结果。


In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(6.5, 4.5))
ax.boxplot(
    samples,
    patch_artist=True,
    boxprops={"facecolor": "#d2e3fc"},
    medianprops={"color": "#d93025", "linewidth": 2},
)
ax.set(title="订单金额箱线图", ylabel="订单金额（元）", xticks=[1], xticklabels=["全部订单"])
ax.grid(axis="y", alpha=0.2)
fig.tight_layout()
plt.show()


**练一练**：修改一个图表参数，观察分布描述的变化。

在 33.4 的基础箱线图上继续操作：

1. 复制基础图表的绘图代码，给 `ax.boxplot()` 添加 `showmeans=True`，用均值标记与中位数（橙色线）的位置差异，判断订单金额分布是否对称；
2. 再把 `whis` 从默认 1.5 改为 2.0，说明须范围变宽后，被判定为异常点的观察发生了什么变化（变得更加/更加不稀有）。

提示：若当前环境无法弹出图像，可在下方单元格里先打印 `samples` 的中位数与均值进行对比，再完成参数修改部分。


In [ ]:
# 请在下方填写代码
import matplotlib.pyplot as plt

# TODO: 定义下面这个开关，补全 ax.boxplot() 中缺失的参数名。
# 提示：基础图表中已演示过如何显示均值标记及其参数名。
showmeans = "待填写"  # <-- 请改成 True

fig, ax = plt.subplots(figsize=(6.5, 4.5))
ax.boxplot(
    samples,
    patch_artist=True,
    boxprops={"facecolor": "#d2e3fc"},
    medianprops={"color": "#d93025", "linewidth": 2},
    showmeans=showmeans,  # TODO: 补全参数名后，应显示均值标记
)
ax.set(
    title="订单金额箱线图（带均值）", ylabel="订单金额（元）", xticks=[1], xticklabels=["全部订单"]
)
fig.tight_layout()
plt.show()


In [ ]:
# ===== 完整答案 =====
import matplotlib.pyplot as plt
import numpy as np

# 1. showmeans=True 显示均值标记
fig, ax = plt.subplots(figsize=(6.5, 4.5))
ax.boxplot(
    samples,
    patch_artist=True,
    boxprops={"facecolor": "#d2e3fc"},
    medianprops={"color": "#d93025", "linewidth": 2},
    showmeans=True,
)
ax.set(
    title="订单金额箱线图（带均值）", ylabel="订单金额（元）", xticks=[1], xticklabels=["全部订单"]
)
ax.grid(axis="y", alpha=0.2)
fig.tight_layout()
plt.show()

# 2. whis=2.0 扩大须范围后再绘制
fig2, ax2 = plt.subplots(figsize=(6.5, 4.5))
ax2.boxplot(
    samples,
    patch_artist=True,
    boxprops={"facecolor": "#d2e3fc"},
    medianprops={"color": "#d93025", "linewidth": 2},
    whis=2.0,
)
ax2.set(
    title="订单金额箱线图（whis=2.0）",
    ylabel="订单金额（元）",
    xticks=[1],
    xticklabels=["全部订单"],
)
ax2.grid(axis="y", alpha=0.2)
fig2.tight_layout()
plt.show()


## 38.5 进阶变体

在基础图表可读的前提下增加分组、布局、注释或交互。新增编码必须服务于一个明确问题。


In [ ]:
import matplotlib.pyplot as plt

rng_box = np.random.default_rng(30)
groups = [
    rng_box.normal(180, 30, 100),
    rng_box.normal(240, 48, 100),
    rng_box.normal(210, 36, 100),
]
fig, ax = plt.subplots(figsize=(8, 4.5))
boxes = ax.boxplot(
    groups, tick_labels=["办公", "数码", "家居"], patch_artist=True, showmeans=True
)
for patch, color in zip(boxes["boxes"], ["#d2e3fc", "#ceead6", "#feefc3"]):
    patch.set_facecolor(color)
ax.set(title="品类订单金额分布", ylabel="订单金额（元）")
fig.tight_layout()
plt.show()


## 38.6 参数说明

- whis：须范围
- showmeans：均值
- notch：中位数缺口
- patch_artist：填充箱体


## 38.7 结果解读

箱体中线是中位数，箱体覆盖中间50%，须外点是潜在异常而非必然错误。


## 38.8 本章实训：图表只改一个编码

这一组实验专门训练“观察一个结果 → 只改一个变量 → 解释变化”。先运行第一个代码单元格，再运行第二个。


In [ ]:
import matplotlib.pyplot as plt

months = ["1月", "2月", "3月", "4月"]
sales = [120, 150, 138, 190]
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.plot(months, sales, marker="o")
ax.set_title("月度销售额")
ax.set_ylabel("销售额（万元）")
ax.grid(alpha=0.25)
plt.show()


### 38.8.1 第一个结果怎么读

标题、坐标轴和单位让读者知道图表回答什么问题。没有这些文字，图形即使画出来也不完整。

请记录：输入是什么、输出是什么、输出支持了哪一个结论。


In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.bar(months, sales, color="#2563EB")
ax.axhline(
    sum(sales) / len(sales), color="#DC2626", linestyle="--", label="平均值"
)
ax.set_title("月度销售额与平均值")
ax.set_ylabel("销售额（万元）")
ax.legend()
plt.show()


### 38.8.2 第二个结果怎么读

第二个实验把折线改成柱状图，并增加平均线。请说明：哪种图更适合看趋势，哪种图更适合比较单月差异？

迁移任务：把一个输入值、一个字段或一个图表参数换成自己的例子，再用一句话解释变化。


## 38.9 错误恢复：图表能画出但读不懂怎么办

真实数据和真实代码都会出问题。本节先观察问题，再用一个明确的检查或修复步骤恢复运行。


In [ ]:
import matplotlib.pyplot as plt

months = ["1月", "2月", "3月"]
sales = [120, 150, 138]
fig, ax = plt.subplots(figsize=(6, 3))
ax.plot(months, sales, marker="o")
ax.set_title("月度销售额")
ax.set_xlabel("月份")
ax.set_ylabel("销售额（万元）")
ax.grid(alpha=0.25)
plt.show()


### 38.9.1 错误恢复步骤

1. 先看错误类型、字段或数据形状。
2. 判断问题发生在输入、处理中间结果还是输出。
3. 修复后重新检查结果，而不是只让代码不报错。

图形没有报错不等于结果可用。遇到“看不懂”的图，优先补标题、坐标轴、单位和关键参照线。

迁移任务：把示例中的输入换成一组会触发问题的数据，并记录你的修复规则。


## 38.10 易错点提醒

- 把箱体高度理解为样本量
- 机械删除所有须外点
- 小样本仍只看箱线摘要


## 38.11 练习与作业

请使用同一份数据完成下面任务，并说明你选择该图表的原因。完成后补充：图表回答了什么问题、最重要的视觉信号是什么、还有哪些信息无法从图中得出。


## 38.12 独立迁移练习

复制最接近的示例，只修改一种视觉编码，并说明阅读任务如何变化。

先在下面单元格完成自己的版本；需要参考时再回看紧邻的示例或参考实现。


In [ ]:
# 独立迁移练习：把单组箱线扩展为「普通 vs 高价」两组对比
# 【目标】从单分布到多分布对比，练习读出两组之间的中心与离散差异。
import matplotlib.pyplot as plt

# 起点示例(已可运行)：按 260 元切分样本，画两组箱线对比。
#   - 用 boxplot(传入列表) 一次画多组；
#   - tick_labels 给每组命名，boxprops/medianprops 统一配色。
regular = samples[samples < 260]
premium = samples[samples >= 260]
fig, ax = plt.subplots(figsize=(7, 4.5))
ax.boxplot(
    [regular, premium],
    patch_artist=True,
    tick_labels=["普通订单", "高价订单"],
    boxprops={"facecolor": "#d2e3fc"},
    medianprops={"color": "#d93025", "linewidth": 2},
)
ax.set(title="普通与高价订单金额分布", ylabel="订单金额（元）")
ax.grid(axis="y", alpha=0.2)
fig.tight_layout()
plt.show()

# ---- 反思记录：两组的箱体位置与长度有何不同 ----
change_note = "待填写"
expected_change = "待填写"
observed_change = "运行后填写"
print(f"改动：{change_note}")
print(f"预期：{expected_change}")
print(f"观察：{observed_change}")


In [ ]:
import matplotlib.pyplot as plt

rng_delivery = np.random.default_rng(300)
delivery = [
    rng_delivery.normal(2.6, 0.6, 90),
    rng_delivery.normal(3.4, 0.9, 90),
]
fig, ax = plt.subplots(figsize=(7, 4.2))
ax.boxplot(delivery, tick_labels=["自营", "第三方"], patch_artist=True)
ax.axhline(3, color="#d93025", linestyle="--", label="3天目标")
ax.set(title="配送时长对比", ylabel="天")
ax.legend(frameon=False)
fig.tight_layout()
plt.show()


## 38.13 小结

使用中位数、四分位距和须快速比较分布并标记潜在离群点。


### 38.13.1 你已经掌握

- 判断箱线图（boxplot）的适用场景
- 准备与图表匹配的数据结构
- 从基础图表扩展到分组、注释或交互变体
- 按照业务问题解读图表并说明结论边界


### 38.13.2 关键参数

| 参数 | 作用 |
| --- | --- |
| `whis` | 须范围 |
| `showmeans` | 均值 |
| `notch` | 中位数缺口 |
| `patch_artist` | 填充箱体 |


### 38.13.3 需要注意

- 把箱体高度理解为样本量
- 机械删除所有须外点
- 小样本仍只看箱线摘要


### 38.13.4 完成检查

- [ ] 能判断什么问题适合使用箱线图（boxplot）
- [ ] 能准备符合要求的数据结构
- [ ] 能独立完成基础图表和一个进阶变体
- [ ] 能调整关键参数并解释视觉变化
- [ ] 能根据图表写出有边界的数据结论


### 38.13.5 下一步推荐

把同一图表迁移到另一份数据，先保留同样的编码，再只改变一个维度。比较迁移前后的可读性，并说明哪些结论仍然成立。
